# RiskOptima 2.x Quick Demos

This notebook highlights the newly refactored core pieces: `MarketData`, `Portfolio`, `FactorRiskModel`, `Constraints`, and `run_backtest`.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from riskoptima.core import MarketData, Portfolio
from riskoptima.risk import FactorRiskModel
from riskoptima.optim import Constraints, optimize_min_variance
from riskoptima.backtest import run_backtest, SMACrossStrategy

warnings.filterwarnings("ignore", category=FutureWarning)

TICKERS = ["AAPL", "MSFT", "KO"]
FACTOR_TICKERS = ["SPY", "QQQ"]
START = "2023-01-01"
END = "2024-01-01"


## 1) MarketData + Portfolio
Build a simple `MarketData` object and a lightweight `Portfolio` definition.


In [ ]:
prices = yf.download(TICKERS, start=START, end=END, progress=False, auto_adjust=False)["Close"]
returns = prices.pct_change(fill_method=None).dropna()

market_data = MarketData(prices=prices, returns=returns, calendar="D")

weights = pd.Series([0.4, 0.4, 0.2], index=TICKERS)
benchmark = yf.download("SPY", start=START, end=END, progress=False, auto_adjust=False)["Close"]
portfolio = Portfolio(weights=weights, benchmark=benchmark)

market_data, portfolio


## 2) FactorRiskModel
Estimate factor exposures and build a covariance matrix.


In [ ]:
factor_prices = yf.download(FACTOR_TICKERS, start=START, end=END, progress=False, auto_adjust=False)["Close"]
factor_returns = factor_prices.pct_change(fill_method=None).dropna()

frm = FactorRiskModel(factor_returns=factor_returns).fit(market_data.returns)

frm.exposures.head()


## 3) Constraints + Optimization
Use factor-aware constraints with mean-variance optimization.


In [ ]:
cov = frm.covariance_matrix()
expected_returns = market_data.returns.mean() * 252

constraints = Constraints(
    weight_bounds=(0.05, 0.7),
    factor_bounds={"SPY": (-0.2, 0.6)}
)

weights_opt = optimize_min_variance(
    cov=cov,
    expected_returns=expected_returns,
    constraints=constraints,
    factor_exposures=frm.exposures
)

weights_opt


## 4) run_backtest
Run a simple SMA crossover strategy on the same price data.


In [ ]:
strategy = SMACrossStrategy(short_window=20, long_window=50)
equity_curve, weights_history = run_backtest(prices=market_data.prices, strategy=strategy)

equity_curve[["PortfolioValue"]].plot(title="SMA Backtest Equity Curve", figsize=(10, 4))
plt.show()

weights_history.tail()
